# COMP90024 Assignment 2: Data Analysis and Visualization

**Group 57**
- Josh Childs (1549150)
- Enzo Noel (1675346)
- Molly Stow (1356089)
- YU-WEI LIN (1537312)

This notebook aims to address the defined analysis goals by fetching data from the Fission backend and generating relevant visualizations.

In [ ]:
# Basic Imports and Setup
import requests
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns # Added for heatmap
from wordcloud import WordCloud # WordCloud import
from datetime import datetime, timedelta

# Load functions from frontend_functions.py
# Ensure frontend_functions.py is in the same directory or Python path
try:
    from frontend_functions import (
        get_wordcloud_data, # This function fetches entity data from /ui/named-entities/label/{label}
        wordcloud_from_data, 
        get_top_each_platform, plot_top_each_platform,
        plot_source_sentiment, 
        plot_sentiment_across_platforms, 
        comparison_plot_keyword_counts, 
        plot_keyword_counts, 
        labels as platform_display_labels, 
        entity_labels,
        fission as FISSION_ROUTER_URL, 
        dataframe # Ensure dataframe function is imported for plot_stacked_bar_sentiment
    )
    print("Successfully imported from frontend_functions.py")
except ImportError as e:
    print(f"Error importing from frontend_functions.py: {e}")
    print("Please ensure frontend_functions.py is in the correct location and all its dependencies (requests, matplotlib, numpy, pandas, wordcloud) are installed in the current Jupyter kernel's environment.")
    FISSION_ROUTER_URL = "http://localhost:9090" 
    # Define dummy dataframe function if import fails for plot_stacked_bar_sentiment
    def dataframe(data, start, end):
        print("[Dummy] dataframe function called.")
        # Create a minimal DataFrame to prevent errors, assuming data is dict of dicts
        start_dt_obj = datetime.fromisoformat(start)
        end_dt_obj = datetime.fromisoformat(end)
        num_days = (end_dt_obj - start_dt_obj).days
        date_idx = pd.to_datetime([start_dt_obj + timedelta(days=x) for x in range(num_days + 1)])
        try:
            df = pd.DataFrame.from_dict(data, orient='index')
            df.index = pd.to_datetime(df.index, errors='coerce') # Coerce if keys are not perfect dates
            return df.reindex(date_idx).fillna(0)
        except:
            return pd.DataFrame(index=date_idx).fillna(0) # Return empty DF with date index
    platform_display_labels = {"openaus": "Open Australia", "bluesky": "Bluesky", "reddit": "Reddit"}
    entity_labels = {"ORG": "Organisations", "PER": "People", "LOC": "Locations", "EVENT": "Events"}
    # Define dummy functions if the import fails
    def get_wordcloud_data(label): print(f"[Dummy] get_wordcloud_data called for {label}"); return {p: {{'dummy_entity':1}} for p in COMMON_PLATFORMS} # Return dict of dict
    def wordcloud_from_data(label, data, **kwargs): print(f"[Dummy] wordcloud_from_data called for {label}")
    def get_top_each_platform(data, count=10, normalise=False): print("[Dummy] get_top_each_platform called."); return {p: [('dummy_entity', 1)] for p in COMMON_PLATFORMS} # Return dict of list of tuples
    def plot_top_each_platform(data, label, normalised=True): print(f"[Dummy] plot_top_each_platform called for {label}.")
    def plot_source_sentiment(table, start, end, data=None, percentages=False): print(f"[Dummy] plot_source_sentiment called."); return {}
    def plot_sentiment_across_platforms(keyword_list, keyword_type, results=None, fission_url=None): print(f"[Dummy] plot_sentiment_across_platforms for {keyword_list}."); return {}
    def comparison_plot_keyword_counts(platforms, start, keyword, data=None, normalise=True): print(f"[Dummy] comparison_plot_keyword_counts for {keyword}."); return {}

# Load autoreload extension for development convenience
%load_ext autoreload
%autoreload 2

# Common parameters (can be adjusted for different analyses)
DEFAULT_START_DATE = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d") # Default to last 30 days
DEFAULT_END_DATE = datetime.now().strftime("%Y-%m-%d")
COMMON_PLATFORMS = ["reddit", "bluesky", "openaus"]

## Analysis Goal 1: What are people talking about? (topics across platforms)

**Tasks:**
- Word cloud for Bluesky, Reddit, and debate topics (OpenAustralia).
- Compare top 10 topics/entities for each platform, including a heatmap for locations.

In [ ]:
# --- Goal 1: Word Clouds and Top Entities ---
print(f"\n--- Running Analysis Goal 1: Topics and Entities ---")

entity_types_for_wc = ["ORG", "PER", "LOC"]
for entity_type_wc in entity_types_for_wc:
    print(f"\nFetching data for {entity_type_wc} word clouds...")
    # Use get_wordcloud_data as it's the function defined in frontend_functions.py for fetching entity data
    wc_data = get_wordcloud_data(entity_type_wc)
    if wc_data:
        try:
            # Assuming wordcloud_from_data is defined in frontend_functions.py
            # and handles the case where data[s] might be empty for a platform.
            wordcloud_from_data(entity_type_wc, wc_data) 
        except ValueError as e:
            if "Only supported for TrueType fonts" in str(e) or "cannot open resource" in str(e):
                print(f"ERROR generating word cloud for {entity_type_wc}: {e}")
                print("This usually means a suitable TrueType Font (.ttf) was not found or specified for the wordcloud library.")
                print("To fix: 1. Ensure a default font (like DejaVuSans.ttf) is accessible by matplotlib/wordcloud on your system.")
                print("         2. Or, modify 'wordcloud_from_data' in 'frontend_functions.py' to specify a 'font_path' in the WordCloud object, e.g., WordCloud(font_path='/path/to/your/font.ttf', ...)")
            else:
                print(f"An unexpected ValueError occurred while generating word cloud for {entity_type_wc}: {e}")
                # import traceback
                # traceback.print_exc() # Uncomment for full traceback if needed
        except Exception as e_general:
            print(f"An unexpected error occurred while generating word cloud for {entity_type_wc}: {e_general}")
            # import traceback
            # traceback.print_exc() # Uncomment for full traceback if needed
    else:
        print(f"Could not fetch {entity_type_wc} data for word clouds.")

entity_types_to_plot_top = ["ORG", "PER", "LOC"]
for entity_type in entity_types_to_plot_top:
    print(f"\nProcessing top entities for type: {entity_type} ({entity_labels.get(entity_type, entity_type)})")
    # Use get_wordcloud_data to fetch raw entity data as it calls the correct endpoint
    raw_entity_data = get_wordcloud_data(entity_type) 
    if raw_entity_data:
        # get_top_each_platform returns a dict: {'platform': [('entity1', count1), ...]}
        top_entities_by_platform = get_top_each_platform(raw_entity_data, count=10, normalise=False) 
        if top_entities_by_platform:
            # Plot bar charts for top entities per platform
            plot_top_each_platform(top_entities_by_platform, entity_labels.get(entity_type, entity_type), normalised=False)
            
            # --- Create Heatmap for these top entities (using raw counts) ---
            print(f"\nGenerating Heatmap for Top {entity_labels.get(entity_type, entity_type)}...")
            try:
                # Consolidate all entities that appear in the top list of any platform
                all_entities_in_tops = set()
                for platform_data_list in top_entities_by_platform.values():
                    for entity, count in platform_data_list:
                        all_entities_in_tops.add(entity)
                
                if not all_entities_in_tops:
                    print(f"No top entities found across platforms for {entity_type} to create a heatmap.")
                    continue # Skip to next entity type

                heatmap_pivot_data = {}
                for platform_key, entity_list_tuples in top_entities_by_platform.items():
                    platform_display_name = platform_display_labels.get(platform_key, platform_key)
                    # Create a dict of entity:count for the current platform's top entities
                    current_platform_counts = dict(entity_list_tuples)
                    # For each globally top entity, get its count for the current platform (0 if not in its top list)
                    heatmap_pivot_data[platform_display_name] = {entity: current_platform_counts.get(entity, 0) for entity in all_entities_in_tops}
                
                if heatmap_pivot_data:
                    heatmap_df = pd.DataFrame(heatmap_pivot_data).reindex(list(all_entities_in_tops))
                    heatmap_df = heatmap_df.fillna(0).astype(int) # Ensure all values are numeric and fill NaNs
                    
                    # Optional: Sort rows by total occurrences across platforms for better readability
                    heatmap_df['total_occurrences'] = heatmap_df.sum(axis=1)
                    heatmap_df = heatmap_df.sort_values(by='total_occurrences', ascending=False)
                    heatmap_df_to_plot = heatmap_df.drop(columns='total_occurrences').head(15) # Show top 15 overall for the heatmap

                    if not heatmap_df_to_plot.empty:
                        plt.figure(figsize=(10, max(8, len(heatmap_df_to_plot) * 0.5))) # Adjust height based on number of entities
                        sns.heatmap(heatmap_df_to_plot, annot=True, fmt="d", cmap="viridis", linewidths=.5)
                        plt.title(f"Top Mentioned {entity_labels.get(entity_type, entity_type)} Across Platforms (Counts Heatmap)", fontsize=16)
                        plt.xlabel("Platform", fontsize=12)
                        plt.ylabel(f"{entity_labels.get(entity_type, entity_type)} Entity", fontsize=12)
                        plt.xticks(rotation=45, ha='right')
                        plt.yticks(rotation=0)
                        plt.tight_layout()
                        plt.show()
                    else:
                        print(f"No data to plot heatmap for {entity_type} after processing.")
                else:
                    print(f"Could not prepare data for heatmap for {entity_type} (heatmap_pivot_data is empty).")
            except Exception as e_heatmap:
                print(f"Error generating heatmap for {entity_type}: {e_heatmap}")
                import traceback
                traceback.print_exc()
        else:
            print(f"Could not get top entities for type: {entity_type}")
    else:
        print(f"Could not fetch raw entity data for type: {entity_type}")

## Analysis Goal 2: How do the overall sentiments compare? (sentiment across platforms and time)

**Tasks:**
- Sentiment stacked bar chart across time for Bluesky, Reddit, and debates (OpenAustralia).

In [ ]:
# --- Goal 2: Overall Sentiment Comparison (Stacked Bar Charts) ---
print(f"\n--- Running Analysis Goal 2: Overall Sentiment Comparison ---")

goal2_start_date = DEFAULT_START_DATE
goal2_end_date = DEFAULT_END_DATE

print(f"Fetching overall sentiment data from {goal2_start_date} to {goal2_end_date}...")

raw_overall_sentiment_data = None
try:
    response = requests.get(
        url=f"{FISSION_ROUTER_URL}/ui/sentiment/keyword/*/start/{goal2_start_date}/end/{goal2_end_date}",
        timeout=1500 
    )
    response.raise_for_status()
    raw_overall_sentiment_data = response.json()
    print("Successfully fetched overall sentiment data.")
except Exception as e:
    print(f"Error fetching overall sentiment data: {e}")
    if 'response' in locals() and response is not None:
        print(f"Response text: {response.text}")

def plot_stacked_bar_sentiment(platform_name, daily_sentiment_data, start_str, end_str):
    """Plots daily sentiment as a stacked bar chart (pos, neu, neg proportions)."""
    if not daily_sentiment_data:
        print(f"No sentiment data for {platform_display_labels.get(platform_name, platform_name)} to plot.")
        return
    
    # Use dataframe function from frontend_functions.py to ensure all dates are present
    # and that index is datetime type for proper plotting.
    df = dataframe(daily_sentiment_data, start_str, end_str) 
    df = df.fillna(0) # Fill NaNs for dates with no data
    df.index = pd.to_datetime(df.index, errors='coerce') # Ensure index is datetime, coerce errors

    required_cols = ['neg', 'neu', 'pos']
    if not all(col in df.columns for col in required_cols):
        print(f"Data for {platform_display_labels.get(platform_name, platform_name)} is missing required sentiment columns {required_cols} or is empty.")
        return

    df_proportions = df[required_cols].copy()
    df_proportions['total_sentiment_values'] = df_proportions[required_cols].sum(axis=1)

    for col in required_cols:
        # Calculate proportion; if total is 0, distribute 1/3 to neutral to avoid division by zero and show something.
        df_proportions[col] = df_proportions.apply(lambda row: row[col] / row['total_sentiment_values'] if row['total_sentiment_values'] > 0 else (1/3 if col=='neu' else 0), axis=1)
    
    plt.figure(figsize=(15, 6))
    # Plot in order: Positive, Neutral, Negative for typical stacked bar visualization
    df_proportions[['pos', 'neu', 'neg']].plot(kind='bar', stacked=True, 
                                               color=['forestgreen', 'wheat', 'firebrick'],
                                               width=0.9, ax=plt.gca())
    
    plt.title(f"Daily Sentiment Proportions on {platform_display_labels.get(platform_name, platform_name)}\n({start_str} to {end_str})")
    plt.xlabel("Date")
    plt.ylabel("Proportion of Sentiments")
    
    ax = plt.gca()
    # Create tick positions and labels, handling potential NaT in index robustly
    # Filter out NaT from index before creating labels and positions
    valid_index = df.index.dropna()
    if not valid_index.empty:
        tick_labels = [item.strftime('%Y-%m-%d') for item in valid_index]
        # Map original index positions to new valid_index positions for xticks
        original_indices = df.index.tolist()
        tick_positions = [original_indices.index(date_val) for date_val in valid_index if date_val in original_indices]
        
        if len(tick_labels) > 20: 
            step = max(1, len(tick_labels) // 10) 
            ax.set_xticks(np.array(tick_positions)[::step])
            ax.set_xticklabels(np.array(tick_labels)[::step], rotation=45, ha='right')
        elif tick_labels: # Ensure there are labels to set
            ax.set_xticks(tick_positions)
            ax.set_xticklabels(tick_labels, rotation=45, ha='right')
        else:
            ax.set_xticks([]) 
            ax.set_xticklabels([])
            print(f"Warning: No valid date ticks could be generated for {platform_display_labels.get(platform_name, platform_name)} chart.")
    else: # No valid dates in index at all
        ax.set_xticks([])
        ax.set_xticklabels([])
        print(f"Warning: Could not generate date ticks for {platform_display_labels.get(platform_name, platform_name)} chart due to invalid date index.")

    plt.legend(title='Sentiment', loc='upper left', bbox_to_anchor=(1, 1))
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.show()

if raw_overall_sentiment_data:
    for platform in COMMON_PLATFORMS:
        if platform in raw_overall_sentiment_data:
            print(f"\nPlotting stacked bar sentiment for {platform_display_labels.get(platform, platform)}...")
            plot_stacked_bar_sentiment(platform, raw_overall_sentiment_data[platform], goal2_start_date, goal2_end_date)
        else:
            print(f"No overall sentiment data found for {platform}.")
else:
    print("Skipping Goal 2 plots due to data fetching error.")

## Analysis Goal 3: How do the sentiments of discussions on topics and on (or by) parties or people compare?

**Tasks:**
- Average sentiment over discussions (in debates and online) about key topics (e.g., housing).
- Average sentiment over discussions (in debates and online) about and BY key people and parties.

In [ ]:
# --- Goal 3: Sentiment of Discussions on Topics/People/Parties ---
print(f"\n--- Running Analysis Goal 3: Sentiment of Discussions ---")

goal3_start_date = DEFAULT_START_DATE
goal3_end_date = DEFAULT_END_DATE

key_topic_goal3 = "housing"
key_people_goal3 = ["Anthony Albanese", "Peter Dutton"] 
key_parties_goal3 = ["Labor Party", "Liberal Party"] 

print(f"\nFetching sentiment for topic: '{key_topic_goal3}'")
print("NOTE: The following calls to '/ui/sentiment-averager' may fail if the backend Fission function ('ui-keywords-sentiment-averager')")
print("      is not correctly implemented to handle the 'type' parameter (e.g., 'topic', 'people', 'parties') in its URL path")
print("      or if the backend logic for that type is missing or has errors.")
print("      If you see 'Invalid type provided' or 'error sending request to function', please check the Fission function logs for 'ui-keywords-sentiment-averager'.")

topic_sentiment_results = plot_sentiment_across_platforms(
    keyword_list=[key_topic_goal3],
    keyword_type="topic" 
)

print(f"\nFetching sentiment for people: {key_people_goal3}")
people_sentiment_results = plot_sentiment_across_platforms(
    keyword_list=key_people_goal3,
    keyword_type="people" 
)

print(f"\nFetching sentiment for parties: {key_parties_goal3}")
parties_sentiment_results = plot_sentiment_across_platforms(
    keyword_list=key_parties_goal3,
    keyword_type="parties" 
)

print("\nGoal 3 Note: The plots above show sentiment of discussions ABOUT the keywords.")
print("Analyzing sentiment OF speeches BY specific people/parties would require a dedicated backend function and data source (e.g., OpenAustralia transcripts filtered by speaker).")

## Analysis Goal 4: Do people discuss the recent topics of debates?

**Tasks:**
- Choose a topic (e.g., housing).
- Graph over time Bluesky/Reddit posts with that topic AND debates with that topic.
- See if discussions rise immediately after (maybe start from a specific date of a debate with a clear topic?).

In [ ]:
# --- Goal 4: Discussion of Debate Topics ---
print(f"\n--- Running Analysis Goal 4: Discussion of Debate Topics ---")

goal4_topic = "housing"
goal4_start_date = (datetime.now() - timedelta(days=90)).strftime("%Y-%m-%d") 
goal4_end_date = DEFAULT_END_DATE

print(f"Plotting counts for topic '{goal4_topic}' from {goal4_start_date} to {goal4_end_date} across platforms.")
counts_data = comparison_plot_keyword_counts(
    platforms=COMMON_PLATFORMS, 
    start=goal4_start_date, 
    keyword=goal4_topic,
    normalise=True 
)

if counts_data:
    print(f"Successfully plotted counts for topic '{goal4_topic}'.")
else:
    print(f"Could not fetch or plot counts for topic '{goal4_topic}'.")

print("\nGoal 4 Note: To see if discussions rise immediately after a specific debate, you would:")
print("1. Identify a specific date of a debate on the chosen topic.")
print("2. Adjust the start/end dates for the plot to focus around that debate date.")
print("3. Observe the 'OpenAustralia' count line (representing debates) and compare with Reddit/Bluesky lines.")

--- End of Analysis Notebook ---